同样没法在这里跑 GPU，下面的代码未实测。新增的 `make_swizzle`、`make_composed_layout`、对 ComposedLayout 做 `composition`、`cutlass.min` 这几个 API 我最不确定，文末列了备选写法。

先说明两个假设：
- "mask" 我按 **causal mask** 实现（`causal=True` 时第 i 行只看 `j ≤ i` 的 key，即 torch SDPA `is_causal` 的左上对齐）。越界列的 mask 仍保留。左上对齐保证每行至少有 `j=0` 可见，不会出现整行被 mask 的 NaN。
- 2-stage 指 K/V 的 `cp.async` 双缓冲：算第 j 块时，下一块已在后台搬运。

## 一、三个改动的设计

**1. Causal mask：跳过 + 只在对角块逐元素 mask**
- Q tile `bidx` 只需要 `j ≤ ceil((bidx+1)·BQ/BK) - 1` 的 K tile，循环次数 `n_tiles = min(n_tiles_all, ceil((bidx+1)·BQ/BK))`，工作量约减半。
- 完全在对角线下方的 tile 不需要 mask，只有跨对角线的 tile 才进 mask 分支，判据是 `j·BK + BK - 1 > bidx·BQ`。
- 第 0 个 tile 每行至少有 1 个可见元素，所以 `m_i` 此后一直是有限值。即使之后某个 tile 里某些行被整块 mask 掉，`max(m_old, -inf) = m_old`、`alpha = 1`、`p = 0`，不会出 NaN。

**2. 2-stage 流水：每轮只需一个 barrier**

```
j 轮开始：wait_group(0)   → 第 j 块已到（上一轮的 prefetch）
          barrier         → ① 所有线程的 cp.async 数据可见
                            ② 所有 warp 已算完第 j-1 块，stage (j+1)%2 可以覆盖
          issue(j+1)      → 往另一个 stage 发 cp.async，commit
          compute(j)      → 与第 j+1 块的搬运重叠
```
- 无论有没有下一块都 `commit`（空组立即完成），这样 `wait_group(0)` 语义统一。
- **补零挪到发射时做**：越界向量本来就不会被 cp.async 写，所以每个线程在发射时就地把自己那几个越界向量写 0，不用再多一次 barrier；可见性由下一轮开头的 barrier 保证。

**3. Swizzle：消除 ldmatrix 的 8 路 bank conflict**
- 一行 64 个 bf16 = 128B，等于 32 个 bank。ldmatrix 的 8 个 lane 读 8 行的同一个 16B 列块，全落在同 4 个 bank，8 路冲突。
- `Swizzle<3,3,3>`：把行内 16B 块号（元素偏移的 bit 3–5）与 `行号 % 8`（bit 6–8）异或，8 行的同列块被分散到 8 个不同位置，覆盖全部 32 个 bank。
- 一个 16B 向量不会被拆开（`M=3`，低 3 位不动），所以 cp.async 的 16B 搬运不受影响。
- 转置视图 `sVt` 不能自己手写 stride：必须用 `composition(sV.layout, 转置映射)` 从已带 swizzle 的 sV 的 layout 派生，保证 V 的读取与写入用的是同一个 swizzle。


## 三、测试代码的改动（在上一版 `test_flash_attn.py` 上替换/新增）

**1) `reference` 加 causal；`run_kernel` 与缓存键加 `causal / swizzle`：**

```python
def reference(Q, K, V, causal=False):
    S = Q.float() @ K.float().transpose(-1, -2) / math.sqrt(D)
    if causal:                                    # 左上对齐：保留 j <= i
        Nq, Nk = S.shape[-2:]
        keep = torch.ones(Nq, Nk, dtype=torch.bool, device=S.device).tril()
        S = S.masked_fill(~keep, float("-inf"))
    O = torch.softmax(S, dim=-1) @ V.float()
    lse2 = torch.logsumexp(S, dim=-1) * LOG2E
    return O, lse2


def run_kernel(Q, K, V, O_full, M_full, dynamic, causal=False, swizzle=True):
    args = (to_cute(Q, dynamic, 2), to_cute(K, dynamic, 2), to_cute(V, dynamic, 2),
            to_cute(O_full, dynamic, 2), to_cute(M_full, dynamic, 1))
    flags = (causal, swizzle)
    key = ("dyn",) + flags if dynamic else (tuple(Q.shape), tuple(K.shape), tuple(O_full.shape)) + flags
    if key not in _compiled:
        _compiled[key] = cute.compile(FLASH_ATTN_naive(causal=causal, use_swizzle=swizzle), *args)
    _compiled[key](*args)
    torch.cuda.synchronize()
```

**2) `run_case` 增加 `causal=False, swizzle=True` 两个关键字参数**，并做三处替换：
```python
def run_case(name, B, Nq, Nk, *, causal=False, swizzle=True, q_scale=1.0, zero_q=False,
             poison=True, dynamic=True, seed=0, o_atol=2e-2, lse_atol=5e-3):
    ...
    run_kernel(Q, K, V, O_full, M_full, dynamic, causal, swizzle)      # 原来没有后两个参数
    o_ref, lse_ref = reference(Q, K, V, causal)                         # 原来没有 causal
    ...
    print(f"[{tag}] {name:<28} ...")                                    # name 里带上 causal 标记即可
```

**3) 新增 swizzle 等价性测试：** swizzle 只改 smem 地址，不改任何计算顺序，所以结果应当**逐位相同**。这是检验 swizzle 与转置视图 `sVt` 是否一致的最灵敏方法。

```python
def test_swizzle_equiv(B=2, Nq=130, Nk=257, causal=True, dynamic=True):
    gen = torch.Generator(device=DEV)
    gen.manual_seed(1)
    Q, K, V = (make_input(B, n, 1.0, gen, poison=False) for n in (Nq, Nk, Nk))
    Np = (Nq + BQ - 1) // BQ * BQ
    outs = []
    for sw in (False, True):
        O = torch.zeros(B, Np, D, dtype=BF16, device=DEV)
        M = torch.zeros(B, Np, dtype=torch.float32, device=DEV)
        run_kernel(Q, K, V, O, M, dynamic, causal, sw)
        outs.append((O[:, :Nq].clone(), M[:, :Nq].clone()))
    same = torch.equal(outs[0][0], outs[1][0]) and torch.equal(outs[0][1], outs[1][1])
    print(f"[{'PASS' if same else 'FAIL'}] swizzle 开/关 逐位一致 (causal={causal})")
    return same
```

**4) `main()` 里的用例表**（在原有 14 例之外新增）：

```python
cases += [
    # 流水线：tile 数 1 / 2 / 3 / 5，覆盖“无预取”“stage 回绕”
    ("流水:1 tile",           1,  64,   64, {}),
    ("流水:2 tile",           1,  64,  128, {}),
    ("流水:3 tile(回绕)",     1,  64,  192, {}),
    ("流水:5 tile+尾巴",      2,  64,  300, {}),
    # causal
    ("causal 单tile",         1,  64,   64, {"causal": True}),
    ("causal 多tile 方阵",    2, 256,  256, {"causal": True}),
    ("causal 非整除",         2, 130,  257, {"causal": True}),
    ("causal Nq>Nk",          1, 200,   80, {"causal": True}),    # 行 i>=Nk 看全部 key
    ("causal Nq<Nk",          1,  70,  300, {"causal": True}),
    ("causal Nq=1",           1,   1,  128, {"causal": True}),
    ("causal 大幅值",         1, 100,  300, {"causal": True, "q_scale": 8.0}),
    ("causal Q=0",            1,  64,  200, {"causal": True, "zero_q": True}),
    # 无 swizzle 的对照路径也要保持正确
    ("无swizzle 基线",        1, 130,  257, {"swizzle": False}),
    ("无swizzle causal",      1, 130,  257, {"swizzle": False, "causal": True}),
]
# ...跑完 results 之后：
results.append(test_swizzle_equiv(dynamic=dyn))
```

## 四、出错时怎么定位

| 现象 | 最可能的原因 |
|---|---|
| 非 causal 的老用例开始失败 | 流水线：`wait/barrier/commit` 的顺序，或 stage 下标 |
| 只有 tile 数 ≥ 3 的用例错 | stage 回绕：预取覆盖了尚未读完的 stage（barrier 位置错） |
| 只有 tile 数 = 1 错 | 序言的 `wait_group(1)` 与组数不匹配 |
| 无 swizzle 对，有 swizzle 错；等价性测试 FAIL | `sVt` 的 swizzle 与 sV 不一致（转置视图构造错）；或 smem 基址未按 1024B 对齐 |
| 所有 causal 用例错、非 causal 对 | 对角块判据 `j·BK+BK-1 > bidx·BQ`，或 `n_tiles` 的上界 |
| 只有 `Nq>Nk` 的 causal 错 | `row ≥ Nk` 时的 mask（应看到全部 key） |
| 出现 NaN | 某行整块被 mask 且 `m_old` 仍是 `-inf`（说明第 0 个 tile 没被处理或没有有效列） |

## 五、API 不确定时的备选写法

- `cute.make_composed_layout(cute.make_swizzle(3,3,3), 0, base)`：若你的版本把 swizzle 放在 allocator 上，改成 `allocate_tensor(dtype, base, 1024, swizzle=cute.make_swizzle(3,3,3))`。此时 `_vt` 里的 `sV_s.layout` 要确认是否已含 swizzle；先 `print(sK.layout)`（`debug=True`）检查。
- `cute.composition(sV_s.layout, ...)`：若对 ComposedLayout 不支持，可把 swizzle 与基础 layout 分开保存，转置时只对基础 layout 做 `make_layout((D,BK), stride=(1,D))`，再重新套同一个 swizzle。
- `cutlass.min`：不可用时改成 `cap = ...; n_tiles = n_tiles if n_tiles < cap else cap`。
- `(a) & (b)` / `|` 组合 Boolean：若报类型错误，把条件拆成嵌套 `if`。

## 六、预期收益与下一步

- causal 的循环次数约减半，且只有对角块进入逐元素 mask 分支。
- 双缓冲把 `cp.async` 的延迟藏在 mma 之后；swizzle 去掉 ldmatrix 的 bank conflict。可以用 `ncu` 的 `smem bank conflicts` 指标（`l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld`）验证开关 swizzle 前后的差别。
- 还没做、但最值得做的：O 经 smem 整理后做 16B 合并写回；把 `acc_s` 的 `scale` 与 `-m` 合成一次 FMA；将 `BK` 增大到 128 以提高每次 ldmatrix 之后的 mma 复用。

In [ ]:
import math
import torch
import cutlass
import cutlass.cute as cute

D = 64      # 特征维度（swizzle 的参数按 128B 行写死，见 _smem_layout）
BQ = 64     # Q tile 行数
BK = 64     # K/V tile 行数
STG = 2     # K/V 流水级数
LOG2E = math.log2(math.e)
HAS_CUDA = torch.cuda.is_available()


class FLASH_ATTN_naive:
    """O = softmax(QK^T/√d [+causal]) V，同时输出 base-2 的 LSE。
       causal=True ：第 i 行只看 j<=i（左上对齐）；
       use_swizzle ：smem 用 128B swizzle（False 仅用于对照调试）；
       K/V 走 2-stage cp.async 双缓冲。"""

    def __init__(self, causal: bool = False, use_swizzle: bool = True,
                 use_pred: bool = True, zero_fill: bool = True,
                 debug: bool = False, num_threads: int = 128):
        self.causal, self.use_swizzle = causal, use_swizzle
        self.use_pred, self.zero_fill, self.debug = use_pred, zero_fill, debug
        self.BQ, self.BK, self.D, self.VEC = BQ, BK, D, 8
        assert D == 64, "Swizzle<3,3,3> 对应 128B 行（64 个 bf16）；其他 D 需要改 swizzle 参数"
        assert BK % 16 == 0

        self.qk_scale = LOG2E / math.sqrt(D)       # exp(x/√d) = 2^(x*qk_scale)

        self.WM = BQ // 16                         # warp 沿 M；不切 N、不切 K
        self.NT = num_threads
        assert self.NT == self.WM * 32

        self.thr_n = D // self.VEC
        self.thr_m = self.NT // self.thr_n
        assert BQ % self.thr_m == 0 and BK % self.thr_m == 0

    # ------------------------------------------------------------ smem layout
    def _smem_layout(self, rows, stages=None):
        """(rows, D[, stages])，行主序；可选套 128B swizzle。"""
        if stages is None:
            base = cute.make_layout((rows, self.D), stride=(self.D, 1))
        else:
            base = cute.make_layout((rows, self.D, stages),
                                    stride=(self.D, 1, rows * self.D))
        if self.use_swizzle:
            # Swizzle<B=3, M=3, S=3>：以元素为单位，bit[3:6) ^= bit[6:9)
            # 即 16B 块号 ^= 行号%8；stage stride=4096 元素 > bit 9，不受影响
            return cute.make_composed_layout(cute.make_swizzle(3, 3, 3), 0, base)
        return base

    def _vt(self, sV_s):
        """sV_s: (BK, D) 的（可能带 swizzle 的）smem tensor → 转置视图 (D, BK)。
        用 composition 从 sV 自己的 layout 派生，保证 swizzle 一致：
          Vt(n,k) = V(k,n)，把 (n,k) 看成 V 的列优先线性下标 n*BK + k。"""
        return cute.make_tensor(
            sV_s.iterator,
            cute.composition(sV_s.layout, cute.make_layout((self.D, self.BK), stride=(self.BK, 1))))

    # ------------------------------------------------------------ gmem → smem
    @cute.jit
    def _g2s(self, tiled_copy, tidx, g_tile, s_tile, c_tile, n_valid):
        """cp.async 搬一个 tile，越界行（全局行号 >= n_valid）不搬；
        zero_fill 时就地给这些越界向量写 0。
        越界向量从不被 cp.async 触碰，所以发射时写 0 不会与异步拷贝冲突；
        对其他线程的可见性由之后的 barrier 保证。"""
        thr = tiled_copy.get_slice(tidx)
        tg = thr.partition_S(g_tile)
        ts = thr.partition_D(s_tile)
        tc = thr.partition_S(c_tile)
        n_rep = tg.shape[0][1]
        n_m = cute.size(tg, mode=[1])
        n_k = cute.size(tg, mode=[2])
        if self.use_pred:
            pred = cute.make_rmem_tensor((n_rep, n_m, n_k), cutlass.Boolean)
            for rv in cutlass.range_constexpr(n_rep):
                for m in cutlass.range_constexpr(n_m):
                    for k in cutlass.range_constexpr(n_k):
                        pred[rv, m, k] = tc[(0, rv), m, k][0] < n_valid
            cute.copy(tiled_copy, tg, ts, pred=pred)
        else:
            cute.copy(tiled_copy, tg, ts)
        if self.zero_fill:
            zero = cutlass.BFloat16(0.0)
            for rv in cutlass.range_constexpr(n_rep):
                for m in cutlass.range_constexpr(n_m):
                    for k in cutlass.range_constexpr(n_k):
                        if tc[(0, rv), m, k][0] >= n_valid:
                            for e in cutlass.range_constexpr(self.VEC):
                                ts[(e, rv), m, k] = zero

    # ------------------------------------------------------------ kernel
    @cute.kernel
    def kernel(self, Q: cute.Tensor, K: cute.Tensor, V: cute.Tensor,
               O: cute.Tensor, M: cute.Tensor,
               tiled_copy_g2s: cute.TiledCopy, tiled_mma: cute.TiledMma):

        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        BQ_, BK_, D_ = self.BQ, self.BK, self.D
        NEG_INF = cutlass.Float32(-float("inf"))
        qk_scale = cutlass.Float32(self.qk_scale)

        gsrcQ = Q[(bidy, None, None)]              # (Nq, D)
        gsrcK = K[(bidy, None, None)]              # (Nk, D)
        gsrcV = V[(bidy, None, None)]              # (Nk, D)
        gsrcO = O[(bidy, None, None)]
        gM = M[(bidy, None)]
        Nq = gsrcQ.shape[0]
        Nk = gsrcK.shape[0]

        # ================= smem：sQ 1 份；sK、sV 各 STG 份 =================
        # 1024B 对齐：swizzle 的一个周期是 8 行×128B，基址对齐才能让 bank 分布与理论一致
        smem = cutlass.utils.SmemAllocator()
        sQ = smem.allocate_tensor(Q.element_type, self._smem_layout(BQ_), 1024)
        sK = smem.allocate_tensor(K.element_type, self._smem_layout(BK_, STG), 1024)   # (BK, D, STG)
        sV = smem.allocate_tensor(V.element_type, self._smem_layout(BK_, STG), 1024)
        if self.debug:
            print("sK layout:", sK.layout)

        # ================= gmem 视图 =================
        gQ = cute.local_tile(gsrcQ, (BQ_, D_), (bidx, 0))
        gO = cute.local_tile(gsrcO, (BQ_, D_), (bidx, 0))
        idQ = cute.make_identity_tensor(gsrcQ.shape)
        idK = cute.make_identity_tensor(gsrcK.shape)
        cQ = cute.local_tile(idQ, (BQ_, D_), (bidx, 0))

        # ================= MMA 划分（用 stage 0 的静态切片定形状）=================
        thr_mma = tiled_mma.get_slice(tidx)
        sK0 = sK[(None, None, 0)]
        sVt0 = self._vt(sV[(None, None, 0)])

        tCsA = thr_mma.partition_A(sQ)
        tCrA = tiled_mma.make_fragment_A(tCsA)                     # Q fragment：常驻寄存器
        tCrB = tiled_mma.make_fragment_B(thr_mma.partition_B(sK0))
        acc_s = cute.make_rmem_tensor(thr_mma.partition_shape_C((BQ_, BK_)), cutlass.Float32)

        tOrP = cute.make_rmem_tensor(thr_mma.partition_shape_A((BQ_, BK_)), Q.element_type)
        tOrV = tiled_mma.make_fragment_B(thr_mma.partition_B(sVt0))
        acc_o = cute.make_rmem_tensor(thr_mma.partition_shape_C((BQ_, D_)), cutlass.Float32)
        acc_o.fill(0.0)

        tCcS = thr_mma.partition_C(cute.make_identity_tensor((BQ_, BK_)))
        tCcO = thr_mma.partition_C(cute.make_identity_tensor((BQ_, D_)))

        NS = cute.size(acc_s, mode=[2])
        NO = cute.size(acc_o, mode=[2])
        KS = cute.size(tCrA, mode=[2])
        KO = cute.size(tOrP, mode=[2])

        # ================= s2r：ldmatrix =================
        atom_A = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4), Q.element_type)
        atom_B = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4), K.element_type)
        atom_Bt = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=True, num_matrices=4), V.element_type)
        s2r_A = cute.make_tiled_copy_A(atom_A, tiled_mma)
        s2r_B = cute.make_tiled_copy_B(atom_B, tiled_mma)
        s2r_Bt = cute.make_tiled_copy_B(atom_Bt, tiled_mma)
        thr_A = s2r_A.get_slice(tidx)
        thr_B = s2r_B.get_slice(tidx)
        thr_Bt = s2r_Bt.get_slice(tidx)

        tAsA = thr_A.partition_S(sQ)
        tArA = thr_A.retile(tCrA)
        tBrB = thr_B.retile(tCrB)          # 寄存器侧只需建一次；smem 侧每轮按 stage 重新 partition
        tVrV = thr_Bt.retile(tOrV)

        # ================= 行统计量 =================
        m_i = cute.make_rmem_tensor((2,), cutlass.Float32)
        l_i = cute.make_rmem_tensor((2,), cutlass.Float32)
        m_i.fill(NEG_INF)
        l_i.fill(0.0)

        # ================= 本 block 需要处理的 K tile 数 =================
        n_tiles = (Nk + BK_ - 1) // BK_
        if self.causal:
            # 本 Q tile 最后一行的行号 = bidx*BQ+BQ-1，最多需要 key 列 0..bidx*BQ+BQ-1
            n_tiles = cutlass.min(n_tiles, ((bidx + 1) * BQ_ + BK_ - 1) // BK_)

        # ================= 序言：Q 与第 0 块 K/V =================
        self._g2s(tiled_copy_g2s, tidx, gQ, sQ, cQ, Nq)
        cute.arch.cp_async_commit_group()                                   # 组 0：Q
        self._g2s(tiled_copy_g2s, tidx,
                  cute.local_tile(gsrcK, (BK_, D_), (0, 0)), sK[(None, None, 0)],
                  cute.local_tile(idK, (BK_, D_), (0, 0)), Nk)
        self._g2s(tiled_copy_g2s, tidx,
                  cute.local_tile(gsrcV, (BK_, D_), (0, 0)), sV[(None, None, 0)],
                  cute.local_tile(idK, (BK_, D_), (0, 0)), Nk)
        cute.arch.cp_async_commit_group()                                   # 组 1：K0、V0
        cute.arch.cp_async_wait_group(1)                                    # 只等 Q
        cute.arch.barrier()
        for k in cutlass.range_constexpr(KS):                               # Q → 寄存器，之后不再碰 sQ
            cute.copy(s2r_A, tAsA[None, None, k], tArA[None, None, k])

        # ================= 主循环 =================
        for j in cutlass.range(n_tiles, unroll=1):
            stage = j % STG

            # (0) 等第 j 块到齐；barrier 同时保证“所有 warp 已算完第 j-1 块”
            cute.arch.cp_async_wait_group(0)
            cute.arch.barrier()

            # (1) 预取第 j+1 块到另一个 stage（该 stage 刚被第 j-1 块用完）
            if j + 1 < n_tiles:
                nxt = (j + 1) % STG
                self._g2s(tiled_copy_g2s, tidx,
                          cute.local_tile(gsrcK, (BK_, D_), (j + 1, 0)), sK[(None, None, nxt)],
                          cute.local_tile(idK, (BK_, D_), (j + 1, 0)), Nk)
                self._g2s(tiled_copy_g2s, tidx,
                          cute.local_tile(gsrcV, (BK_, D_), (j + 1, 0)), sV[(None, None, nxt)],
                          cute.local_tile(idK, (BK_, D_), (j + 1, 0)), Nk)
            cute.arch.cp_async_commit_group()      # 无论是否发射都提交（空组立即完成），wait 语义统一

            # (2) 当前 stage 的 smem 视图
            sK_s = sK[(None, None, stage)]
            sVt_s = self._vt(sV[(None, None, stage)])
            tBsB = thr_B.partition_S(sK_s)
            tVsV = thr_Bt.partition_S(sVt_s)

            # (3) S = Q K_j^T
            acc_s.fill(0.0)
            for k in cutlass.range_constexpr(KS):
                cute.copy(s2r_B, tBsB[None, None, k], tBrB[None, None, k])
                cute.gemm(tiled_mma, acc_s, tCrA[None, None, k], tCrB[None, None, k], acc_s)

            # (4) mask：越界列 +（causal）对角线上方
            need_mask = (j * BK_ + BK_) > Nk
            if self.causal:
                need_mask = need_mask | ((j * BK_ + BK_ - 1) > (bidx * BQ_))
            if need_mask:
                for h in cutlass.range_constexpr(2):
                    for n in cutlass.range_constexpr(NS):
                        for e in cutlass.range_constexpr(2):
                            col_g = j * BK_ + tCcS[(e, h), 0, n][1]
                            masked = col_g >= Nk
                            if self.causal:
                                row_g = bidx * BQ_ + tCcS[(e, h), 0, n][0]
                                masked = masked | (col_g > row_g)
                            if masked:
                                acc_s[(e, h), 0, n] = NEG_INF

            # (5) online safe softmax（base-2）
            for h in cutlass.range_constexpr(2):
                local_max = NEG_INF
                for n in cutlass.range_constexpr(NS):
                    for e in cutlass.range_constexpr(2):
                        local_max = cute.arch.fmax(local_max, acc_s[(e, h), 0, n])
                local_max = cute.arch.fmax(local_max, cute.arch.shuffle_sync_bfly(local_max, offset=2))
                local_max = cute.arch.fmax(local_max, cute.arch.shuffle_sync_bfly(local_max, offset=1))

                m_old = m_i[h]
                m_new = cute.arch.fmax(m_old, local_max * qk_scale)
                alpha = cute.math.exp2(m_old - m_new, fastmath=True)
                m_i[h] = m_new

                row_sum = cutlass.Float32(0.0)
                for n in cutlass.range_constexpr(NS):
                    for e in cutlass.range_constexpr(2):
                        p = cute.math.exp2(acc_s[(e, h), 0, n] * qk_scale - m_new, fastmath=True)
                        acc_s[(e, h), 0, n] = p
                        row_sum = row_sum + p
                l_i[h] = l_i[h] * alpha + row_sum

                for n in cutlass.range_constexpr(NO):
                    for e in cutlass.range_constexpr(2):
                        acc_o[(e, h), 0, n] = acc_o[(e, h), 0, n] * alpha

            # (6) P：C 片段 → A 片段（寄存器内重排）
            for kb in cutlass.range_constexpr(KO):
                for kh in cutlass.range_constexpr(2):
                    for h in cutlass.range_constexpr(2):
                        for e in cutlass.range_constexpr(2):
                            tOrP[(e, h, kh), 0, kb] = acc_s[(e, h), 0, 2 * kb + kh].to(Q.element_type)

            # (7) O += P V_j
            for k in cutlass.range_constexpr(KO):
                cute.copy(s2r_Bt, tVsV[None, None, k], tVrV[None, None, k])
                cute.gemm(tiled_mma, acc_o, tOrP[None, None, k], tOrV[None, None, k], acc_o)
            # 不再需要循环末尾的 barrier：下一轮开头的 barrier 已覆盖

        # ================= 收尾 =================
        for h in cutlass.range_constexpr(2):
            l = l_i[h]
            l = l + cute.arch.shuffle_sync_bfly(l, offset=2)
            l = l + cute.arch.shuffle_sync_bfly(l, offset=1)
            inv_l = cutlass.Float32(1.0) / l
            for n in cutlass.range_constexpr(NO):
                for e in cutlass.range_constexpr(2):
                    acc_o[(e, h), 0, n] = acc_o[(e, h), 0, n] * inv_l

            row = bidx * BQ_ + tCcO[(0, h), 0, 0][0]
            if (tidx % 4 == 0) & (row < Nq):
                gM[row] = m_i[h] + cute.math.log2(l)       # base-2 LSE

        tCgO = thr_mma.partition_C(gO)
        for h in cutlass.range_constexpr(2):
            row = bidx * BQ_ + tCcO[(0, h), 0, 0][0]
            if row < Nq:
                for n in cutlass.range_constexpr(NO):
                    for e in cutlass.range_constexpr(2):
                        tCgO[(e, h), 0, n] = acc_o[(e, h), 0, n].to(O.element_type)

    # ------------------------------------------------------------ host
    @cute.jit
    def __call__(self, Q: cute.Tensor, K: cute.Tensor, V: cute.Tensor,
                 O: cute.Tensor, M: cute.Tensor):
        assert Q.element_type == cutlass.BFloat16
        B = Q.shape[0]
        Nq, Nk, Nv, No, Nm = Q.shape[1], K.shape[1], V.shape[1], O.shape[1], M.shape[1]

        src3_Q = cute.make_tensor(Q.iterator, cute.make_layout((B, Nq, self.D), stride=(Nq * self.D, self.D, 1)))
        src3_K = cute.make_tensor(K.iterator, cute.make_layout((B, Nk, self.D), stride=(Nk * self.D, self.D, 1)))
        src3_V = cute.make_tensor(V.iterator, cute.make_layout((B, Nv, self.D), stride=(Nv * self.D, self.D, 1)))
        dst3_O = cute.make_tensor(O.iterator, cute.make_layout((B, No, self.D), stride=(No * self.D, self.D, 1)))
        dst3_M = cute.make_tensor(M.iterator, cute.make_layout((B, Nm), stride=(Nm, 1)))

        copy_atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(), Q.element_type, num_bits_per_copy=128)
        tiled_copy_g2s = cute.make_tiled_copy_tv(
            copy_atom,
            cute.make_layout((self.thr_m, self.thr_n), stride=(self.thr_n, 1)),
            cute.make_layout((1, self.VEC)))

        mma_atom = cute.make_mma_atom(
            cute.nvgpu.warp.MmaF16BF16Op(cutlass.BFloat16, cutlass.Float32, (16, 8, 16)))
        tiled_mma = cute.make_tiled_mma(mma_atom, cute.make_layout((self.WM, 1, 1)))

        # smem：sQ(8KB) + sK(2×8KB) + sV(2×8KB) = 40KB，再加 1KB 给 1024B 对齐的余量
        smem_bytes = (self.BQ + 2 * STG * self.BK) * self.D * 2 + 1024
        self.kernel(src3_Q, src3_K, src3_V, dst3_O, dst3_M, tiled_copy_g2s, tiled_mma).launch(
            grid=((Nq + self.BQ - 1) // self.BQ, B, 1),
            block=(self.NT, 1, 1),
            smem=smem_bytes,
        )